### Docker


Empaqueta aplicaciones y sus dependencias en contenedores. utiliza imagenes "plantillas" para crear los contenedores

### Compose

permite la gestion de todos los servicios, redes y volumennes en unico archivo de configuracion yAML

Se crean 3 archivos el 
1. Dockerfile. es un script que le indica a Docker cómo construir tu imagen Docker. 
2. El requirements.txt Alli se le indica cuales son las dependencias que necesita el proyecto y su version. (colocarla siempre)
3. El compose.yml Permite gestionar aplicaciones con múltiples contenedores. Aquí, definiremos tanto un contenedor Django como un contenedor de base de datos PostgreSQL.

#### En Dockerfile. 
- ENV PYTHONDONTWRITEBYTECODE 1 Evita que python genere archivos .pyc (bytecode compilado) dentro del contenedor. con el se evitan problemas de permisos de archivos y mantieneel contenedor ordenado sin archivos temporales.
- ENV PYTHONUNBUFFERED 1 Esta variable asegura que la salida de Python (como los logs de Django) se envíe directamente a la terminal del contenedor en tiempo real, sin pasar por un búfer intermedio. Con éste siempre se ve que esta pasandodentro de laaplicació de forma instantanea.

COPY requirements.txt /app/
RUN python -m pip install --upgrade pip && \
    python -m pip install --no-cache-dir -r requirements.txt

- Instala las dependencias para aprovechar las capaas de docker por eso el orden es asi primero copy .....
- copy instala solo las librerias
- pip instala todo  # "Docker "recuerda" esa capa. Si mañana cambias una línea de tu código de Django pero no añades nuevas librerías, Docker saltará el paso de instalación y usará lo que ya tiene guardado. Se ahorra mucho tiempo."

- --no-cache-dir: Imagen final más pequeña.
- -m pip es incluso más seguro que solo pip porque garantiza que estás usando el instalador exacto de la versión de Python de la imagen
- python -m pip: Ejecuta el módulo pip usando el intérprete de Python actual
- install --upgrade pip: Le indica a pip que descargue e instale la versión más reciente de sí mismo, reemplazando la que tengas actualmente.
- &&: Es un operador lógico de la terminal. Significa que el siguiente comando (el que vendría después) solo se ejecutará si la actualización de pip termina con éxito (sin errores).




### En Requirements.txt

- Django 5.2 LTS Utiliza la version LTS Long term Support por lo qu garantiza soporte  y parches de seguridad hasta 2028.
- Djangorestframework(DRF) 
   -Convierte tus modelos de base de datos a JSON (y viceversa) de forma automática y segura.
   -(Browsable API): Al instalarlo, si entras a una URL desde el navegador, DRF te muestra una interfaz web para probar la API, enviar datos y ver los JSON. Es increíble para el desarrollo.
   -Se configuro en settings.py

- psycopg[binary] es el driver de la base de datos y le da soporte asincrono a django.con éste en settings.py el host no es localhost dino db.

- gunicorn>=23.0.0,<24.0.0 s el que "empuja" a Django para que soporte muchas visitas a la vez.
    - Es el estandar moderno soporta multiples petciones reales, es estable y onsume pocos recursos. hace que no tengamos que usar 
    python manage.py runserver. Este comando es solo para programar; es lento, se bloquea si entran dos personas al mismo tiempo y no es seguro.
    - Se debe configurar en el dDockerfile al final del archivo o en el docker-compose.yml 
    services:
        web:
            build: .
            command: gunicorn mi_proyecto.wsgi:application --bind 0.0.0.0:8000
    --bind 0.0.0.0:8000: Le dice a Gunicorn que escuche en todas las interfaces de red del contenedor en el puerto 8000.

- django-environ>=0.11.2,<1.0.0  sirve para sepaar la configuracion de contraseñas de el codigo fuente.
    -para ello crea un archivo .env este archivo nunca se sube a los repositorios.
    -En el .env pones: DATABASE_URL=postgres://usuario:password@db:5432/nombre_db
    -En el settings.py solo pones: 'default': env.db() Él se encarga de separar cada parte automáticamente. 
    -Convierte de Strings a el tipo de dato de Django.
    -¡NUNCA lo subas a GitHub!: Añade .env a tu archivo .gitignore.
    -5432 es el puerto donde vive la base de datos DENTRO de la red de Docker

- whitenoise[brotli]>=6.8.0,<6.9.0 Es casi obligatorio en Docker para que Django pueda servir sus propios archivos CSS/JS sin necesidad de un servidor Nginx separado (aunque puedes usar ambos). ya que django por seguridad y eficiencia, no sabe entregar archivos CSS, JS o imágenes cuando está en modo producción (DEBUG=False).
    -solo tienes que añadir una línea en la lista de MIDDLEWARE de tu settings.py.
    -Antes de arrancar en producción (especialmente con WhiteNoise), debes ejecutar este comando una sola vez para reunir todos los estilos:
    docker compose exec web python manage.py collectstatic --noinput
    -La extensión [brotli] es una extra que le indica a Python que instale una librería adicional para mejorar la compresión de tus archivos.
        -da Compresión máxima: Al ejecutar collectstatic, WhiteNoise crea una copia comprimida de cada archivo .css y .js usando Brotli.
        -Archivos ultra ligeros: Brotli comprime entre un 15% y 25% más que Gzip.
        -permite que la pagina cargue mas rapido.


### El archivo docker-compose.yml

Un compose.yml archivo permite gestionar aplicaciones con múltiples contenedores. Aquí, definiremos tanto un contenedor Django como un contenedor de base de datos PostgreSQL.

El archivo compose utiliza un archivo de entorno llamado .env, lo que facilitará mantener la configuración separada del código de la aplicación. Las variables de entorno que se enumeran aquí son estándar para la mayoría de las aplicaciones:

### Como ponerlo en marcha

1. Consruir y levantar

In [ ]:
bash
docker-compose up -d --build

2. Crear tablas en la base de datos

In [ ]:
bash
docker-compose exec web python manage.py migrate

3. Crear usuario administrador

In [ ]:
bash
docker compose exec web python manage.py createsuperuser

4. Recolectar archivos estaticos (para WhiteNoise)

In [ ]:
bash
docker compose exec web python manage.py createsuperuser

### Abrir Proyecto
Para abrir el proyecto despues debo tener el Docker-Desktop abierto y ejecutar.

In [ ]:
docker-compose up  # teniendo VSCode abierto.
docker-compose up -d # asi cierre VSCode.

### Bibliografia

- https://www.docker.com/blog/how-to-dockerize-django-app/

- https://github.com/docker/awesome-compose/tree/master/official-documentation-samples/django/

- Creando un proyecto DJANGO con DOCKER. https://www.youtube.com/watch?v=s8po-frUVwg&t=712s 
Sergio Infante.